# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
with open(glob.glob(f"{data_dir}/*.md")[0], "r") as f:
    text = f.read()

In [ ]:
# Step 4: Text Chunking and Dataset Creation

from pathlib import Path
from typing import List
import os

import datasets
from markdown_it import MarkdownIt

from sdg_hub import Flow


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def set_model_config(flow_object: Flow) -> Flow:
    """
    Configure model settings for SDG Hub flows using environment variables.
    """
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")

    if model_provider == "hosted_vllm":
        flow_object.set_model_config(
            model=os.getenv("VLLM_MODEL", "openai/RedHatAI/gpt-oss-20b"),
            api_base=os.getenv("VLLM_API_BASE", "http://inference-server-ai-inference-server-ocp.apps.cluster-s972k.s972k.sandbox1774.opentlc.com/v1"),
            api_key=os.getenv("VLLM_API_KEY", "EMPTY"),
            enable_reasoning=os.getenv("ENABLE_REASONING", "false").lower() in ("1", "true", "yes"),
        )
    elif model_provider == "openai":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    elif model_provider == "openrouter":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
            api_base="https://openrouter.ai/api/v1",
        )
    elif model_provider == "ollama":
        flow_object.set_model_config(
            model=os.getenv("OLLAMA_MODEL", "ollama/gemma2"),
            api_base=os.getenv("OLLAMA_API_BASE", "http://localhost:11434"),
        )
    elif model_provider == "maas":
        flow_object.set_model_config(
            model=os.getenv("MAAS_MODEL"),
            api_base=os.getenv("MAAS_API_BASE"),
            api_key=os.getenv("MAAS_API_KEY"),
        )
    else:
        raise ValueError(f"Unsupported MODEL_PROVIDER: {model_provider}")

    return flow_object


def build_icl_with_sdg_hub(markdown_text: str) -> dict:
    """
    Generate ICL fields with an SDG Hub flow (PromptBuilder + LLM + parsers).
    """
    flow_candidates = [
        Path("icl_generation/flow.yaml"),
        Path("examples/knowledge_tuning/enhanced_summary_knowledge_tuning/icl_generation/flow.yaml"),
    ]
    flow_path = next((path for path in flow_candidates if path.exists()), None)
    if flow_path is None:
        raise FileNotFoundError("ICL flow file not found in expected locations.")

    # Keep prompt size bounded while still representing the processed document.
    icl_input_document = markdown_text.strip()[:20000]
    if not icl_input_document:
        raise ValueError("Processed markdown is empty. Cannot generate ICL.")

    icl_input = datasets.Dataset.from_dict({"document": [icl_input_document]})
    icl_flow = Flow.from_yaml(str(flow_path))
    icl_flow = set_model_config(icl_flow)

    generated_icl = icl_flow.generate(
        icl_input,
        runtime_params={"gen_icl": {"max_tokens": 1536}},
        max_concurrency=1,
    )

    required_fields = [
        "document_outline",
        "icl_document",
        "icl_query_1",
        "icl_query_2",
        "icl_query_3",
        "domain",
    ]

    if len(generated_icl) == 0:
        raise ValueError("ICL generation flow returned no rows.")

    missing_fields = [col for col in required_fields if col not in generated_icl.column_names]
    if missing_fields:
        raise ValueError(f"ICL generation missing required fields: {missing_fields}")

    generated_row = generated_icl[0]
    icl = {}
    for field in required_fields:
        value = generated_row[field]
        if isinstance(value, list):
            value = value[0] if value else ""
        icl[field] = str(value).strip()

    return icl


chunks = chunk_markdown(text, max_tokens=5000, overlap=1000)
if not chunks and text.strip():
    chunks = [text.strip()]


# Prepare seed data for the SDG-Hub knowledge pipeline.
#
# The seed data requires the following fields:
#   - document_outline: A concise title or summary that accurately represents the entire document.
#     For documents covering multiple themes, consider providing multiple outlines (one per section).
#   - icl_document: A representative sample extract from the document. This may include tables, code snippets, definitions, etc.
#   - icl_query_1, icl_query_2, icl_query_3: Three questions based on the icl_document sample.
#   - domain: The domain or subject area of the document.
#
# The code below creates a HuggingFace Dataset from the document chunks,
# then maps the required ICL fields to each entry, and finally saves the result as a JSONL file.

seed_data = datasets.Dataset.from_dict({"document": chunks})
icl = build_icl_with_sdg_hub(text)
if not icl["icl_document"] and chunks:
    icl["icl_document"] = chunks[0][:3000]

# Map the ICL fields to each document chunk (if you want to use the same ICL for all, as shown here)
seed_data = seed_data.map(lambda x: icl)

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("seed_data.jsonl", orient="records", lines=True)

### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook